In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score

C:\Users\AIWM_W01\.conda\envs\Dev_AIWM\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [5]:
base_dir = r"C:\Dev_AIWM\jupyter\KNN\codes\0716\P1_ImpExp"

In [7]:
df = pd.read_csv(os.path.join(base_dir, "Clusters_by_Imps.csv"))
df.set_index('idx_code', inplace=True)
model_cols = {
    'Orig': 'Orig_Cluster_Labels',
    'NonImp': 'NonImp_Cluster_Labels',
    'MedImp': 'MedImp_Cluster_Labels',
    'MICE-I10': 'MICEImp-I10_Cluster_Labels',
    'MICE-I20': 'MICEImp-I20_Cluster_Labels',
    'KNN-K5': 'KNNImp-K5_Cluster_Labels',
    'KNN-K10': 'KNNImp-K10_Cluster_Labels',
    'KNN-K10NW': 'KNNImp-K10NW_Cluster_Labels',
    'KNN-K20': 'KNNImp-K20_Cluster_Labels'
}

print(df.columns.tolist())
print(df.shape)

df

['death_label', 'Orig_Cluster_Labels', 'NonImp_Cluster_Labels', 'MedImp_Cluster_Labels', 'MICEImp-I10_Cluster_Labels', 'MICEImp-I20_Cluster_Labels', 'KNNImp-K5_Cluster_Labels', 'KNNImp-K10_Cluster_Labels', 'KNNImp-K10NW_Cluster_Labels', 'KNNImp-K20_Cluster_Labels']
(1282, 10)


,death_label,Orig_Cluster_Labels,NonImp_Cluster_Labels,MedImp_Cluster_Labels,MICEImp-I10_Cluster_Labels,MICEImp-I20_Cluster_Labels,KNNImp-K5_Cluster_Labels,KNNImp-K10_Cluster_Labels,KNNImp-K10NW_Cluster_Labels,KNNImp-K20_Cluster_Labels
idx_code,,,,,,,,,,
3,0,4,3,4,4,4,3,3,1,3
4,0,4,4,4,4,4,4,4,1,1
44,0,4,4,4,4,4,4,4,4,4
46,1,3,3,4,3,3,3,3,3,3
72,1,2,2,3,2,2,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...
20262,0,4,4,1,4,4,4,4,1,4
20297,1,2,2,3,2,2,2,2,2,2
20312,0,4,4,4,4,4,4,4,4,4


In [8]:
## ARI

ari_rows = []
for model_name, col_name in model_cols.items():
    if model_name == 'Orig':
        ari_val = 1.0
    else:
        ari_val = adjusted_rand_score(df[model_cols['Orig']], df[col_name])

    ari_rows.append({
        'Comparison': f'Orig vs {model_name}',
        'ARI': ari_val
    })

ari_df = pd.DataFrame(ari_rows)

print("[ARI vs Orig]")

ari_df

[ARI vs Orig]


,Comparison,ARI
0,Orig vs Orig,1.000000
1,Orig vs NonImp,0.711946
2,Orig vs MedImp,0.499699
3,Orig vs MICE-I10,0.691932
4,Orig vs MICE-I20,0.691932
5,Orig vs KNN-K5,0.627486
6,Orig vs KNN-K10,0.584035
7,Orig vs KNN-K10NW,0.600971
8,Orig vs KNN-K20,0.592791


In [9]:
# ALL pairwise ARI

model_names = list(model_cols.keys())
ari_matrix = pd.DataFrame(index=model_names, columns=model_names, dtype=float)

for m1 in model_names:
    for m2 in model_names:
        ari_matrix.loc[m1, m2] = adjusted_rand_score(
            df[model_cols[m1]],
            df[model_cols[m2]]
        )

print("\n[Pairwise ARI matrix]")
ari_matrix


[Pairwise ARI matrix]


,Orig,NonImp,MedImp,MICE-I10,MICE-I20,KNN-K5,KNN-K10,KNN-K10NW,KNN-K20
Orig,1.000000,0.711946,0.499699,0.691932,0.691932,0.627486,0.584035,0.600971,0.592791
NonImp,0.711946,1.000000,0.512786,0.674935,0.674935,0.708564,0.618028,0.607578,0.670700
MedImp,0.499699,0.512786,1.000000,0.504172,0.504172,0.500990,0.496405,0.502983,0.485835
MICE-I10,0.691932,0.674935,0.504172,1.000000,1.000000,0.615947,0.576935,0.592185,0.605279
MICE-I20,0.691932,0.674935,0.504172,1.000000,1.000000,0.615947,0.576935,0.592185,0.605279
KNN-K5,0.627486,0.708564,0.500990,0.615947,0.615947,1.000000,0.649783,0.595723,0.684391
KNN-K10,0.584035,0.618028,0.496405,0.576935,0.576935,0.649783,1.000000,0.605281,0.615643
KNN-K10NW,0.600971,0.607578,0.502983,0.592185,0.592185,0.595723,0.605281,1.000000,0.692747
KNN-K20,0.592791,0.670700,0.485835,0.605279,0.605279,0.684391,0.615643,0.692747,1.000000


In [10]:
summary_df = pd.DataFrame({
    'Model': [
        'Orig', 'NonImp', 'MedImp',
        'MICE-I10', 'MICE-I20',
        'KNN-K5', 'KNN-K10', 'KNN-K10NW'
    ],
    'PCs': [6, 6, 6, 6, 6, 6, 6, 6],
    'Explained_variance_pct': [32.80, 32.91, 32.73, 32.85, 32.85, 32.76, 32.76, 32.79],
    'WCSS_k8': [
        53609.32960153667,   # Orig
        68840.02873972207,   # NonImp
        50489.55156630665,   # MedImp
        54466.069531934074,  # MICE-I10
        54201.25985961918,   # MICE-I20
        50309.41860807143,   # KNN-K5
        54915.702380522365,  # KNN-K10
        50668.93948232616    # KNN-K10NW
    ],
    'Silhouette_avg_k8': [
        0.146,  # Orig
        0.162,  # NonImp
        0.128,  # MedImp
        0.122,  # MICE-I10
        0.137,  # MICE-I20
        0.133,  # KNN-K5
        0.135,  # KNN-K10
        0.117   # KNN-K10NW
    ],
    'Silhouette_std_k8': [
        0.271,  # Orig
        0.308,  # NonImp
        0.068,  # MedImp
        0.274,  # MICE-I10
        0.265,  # MICE-I20
        0.085,  # KNN-K5
        0.275,  # KNN-K10
        0.073   # KNN-K10NW
    ]
})

# ARI_vs_Orig 붙이기
ari_map = dict(zip(
    ari_df['Comparison'].str.replace('Orig vs ', '', regex=False),
    ari_df['ARI']
))
ari_map['Orig'] = 1.0

summary_df['ARI_vs_Orig'] = summary_df['Model'].map(ari_map)

print("\n[Summary table]")
summary_df


[Summary table]


,Model,PCs,Explained_variance_pct,WCSS_k8,Silhouette_avg_k8,Silhouette_std_k8,ARI_vs_Orig
0,Orig,6,32.80,53609.329602,0.146,0.271,1.000000
1,NonImp,6,32.91,68840.028740,0.162,0.308,0.711946
2,MedImp,6,32.73,50489.551566,0.128,0.068,0.499699
3,MICE-I10,6,32.85,54466.069532,0.122,0.274,0.691932
4,MICE-I20,6,32.85,54201.259860,0.137,0.265,0.691932
5,KNN-K5,6,32.76,50309.418608,0.133,0.085,0.627486
6,KNN-K10,6,32.76,54915.702381,0.135,0.275,0.584035
7,KNN-K10NW,6,32.79,50668.939482,0.117,0.073,0.600971


In [13]:
## Cluster size tab

def cluster_size_series(labels, name):
    s = pd.Series(labels).value_counts().sort_index()
    s.name = name
    return s

cluster_sizes = pd.concat([
    cluster_size_series(df['Orig_Cluster_Labels'], 'Orig'),
    cluster_size_series(df['NonImp_Cluster_Labels'], 'NonImp'),
    cluster_size_series(df['MedImp_Cluster_Labels'], 'MedImp'),
    cluster_size_series(df['MICEImp-I10_Cluster_Labels'], 'MICE-I10'),
    cluster_size_series(df['MICEImp-I20_Cluster_Labels'], 'MICE-I20'),
    cluster_size_series(df['KNNImp-K5_Cluster_Labels'], 'KNN-K5'),
    cluster_size_series(df['KNNImp-K10_Cluster_Labels'], 'KNN-K10'),
    cluster_size_series(df['KNNImp-K10NW_Cluster_Labels'], 'KNN-K10NW'),
    cluster_size_series(df['KNNImp-K20_Cluster_Labels'], 'KNN-K20')
], axis=1).fillna(0).astype(int)

cluster_sizes.index.name = 'Cluster'

print("\n[Cluster sizes]")
cluster_sizes


[Cluster sizes]


,Orig,NonImp,MedImp,MICE-I10,MICE-I20,KNN-K5,KNN-K10,KNN-K10NW,KNN-K20
Cluster,,,,,,,,,
1,405,392,329,386,386,372,373,542,508
2,322,330,238,329,329,325,347,349,332
3,163,182,353,192,192,195,146,147,198
4,392,378,362,375,375,390,416,244,244


In [14]:
## Cross-tab tables

model_cols = {
    'NonImp': 'NonImp_Cluster_Labels',
    'MedImp': 'MedImp_Cluster_Labels',
    'MICE-I10': 'MICEImp-I10_Cluster_Labels',
    'MICE-I20': 'MICEImp-I20_Cluster_Labels',
    'KNN-K5': 'KNNImp-K5_Cluster_Labels',
    'KNN-K10': 'KNNImp-K10_Cluster_Labels',
    'KNN-K10NW': 'KNNImp-K10NW_Cluster_Labels',
    'KNN-K20': 'KNNImp-K20_Cluster_Labels'
}

cross_tabs = {}

for model_name, col_name in model_cols.items():
    ct = pd.crosstab(df['Orig_Cluster_Labels'], df[col_name])
    cross_tabs[model_name] = ct

    print(f"\n[Cross-tab: Orig vs {model_name}]")
    print(ct)

    ct.to_csv(
        os.path.join(base_dir, f'Crosstab_Orig_vs_{model_name}.csv'),
        encoding='utf-8-sig'
    )


[Cross-tab: Orig vs NonImp]
NonImp_Cluster_Labels    1    2    3    4
Orig_Cluster_Labels                      
1                      348    1    0   56
2                        8  312    2    0
3                        1    8  153    1
4                       35    9   27  321

[Cross-tab: Orig vs MedImp]
MedImp_Cluster_Labels    1    2    3    4
Orig_Cluster_Labels                      
1                      165  230    5    5
2                        0    2  320    0
3                        0    0    6  157
4                      164    6   22  200

[Cross-tab: Orig vs MICE-I10]
MICEImp-I10_Cluster_Labels    1    2    3    4
Orig_Cluster_Labels                           
1                           345    1    0   59
2                             2  317    1    2
3                             0   11  145    7
4                            39    0   46  307

[Cross-tab: Orig vs MICE-I20]
MICEImp-I20_Cluster_Labels    1    2    3    4
Orig_Cluster_Labels                           


In [15]:
print("[Cross-tab: Orig vs MedImp]")
ct_orig_med

[Cross-tab: Orig vs MedImp]


NameError: name 'ct_orig_med' is not defined

In [18]:
print("[Cross-tab: Orig vs NonImp]")
ct_orig_non

[Cross-tab: Orig vs NonImp]


NonImp_Cluster_Labels,1,2,3,4,5,6,7,8
Orig_Cluster_Labels,,,,,,,,
1,347,0,0,0,0,0,6,0
2,0,229,56,0,11,205,0,3200
3,0,0,0,0,0,163,0,1649
4,0,8,383,3,165,2444,1,988
5,0,1223,81,0,0,116,0,34
6,0,168,2618,153,44,1723,0,20
7,0,29,8,22,1468,2,0,7
8,0,569,353,1755,93,6,0,2


In [19]:
print("[Cross-tab: MedImp vs NonImp]")
ct_med_non

[Cross-tab: MedImp vs NonImp]


NonImp_Cluster_Labels,1,2,3,4,5,6,7,8
MedImp_Cluster_Labels,,,,,,,,
1,0,0,7,0,2,69,1,3704
2,4,99,70,2,15,177,0,1422
3,12,2,159,2,86,2402,2,514
4,319,6,54,30,1503,38,3,19
5,9,28,1172,1118,81,216,1,7
6,2,613,257,743,36,11,0,2
7,0,1413,267,18,16,150,0,189
8,1,65,1513,20,42,1596,0,43


In [16]:
# Cluster-wise Jaccard similarity (Cross tab과 같은 설명)

def clusterwise_jaccard(labels_ref, labels_alt, ref_name='Orig', alt_name='Alt'):
    """
    labels_ref: 기준 clustering labels (e.g., Orig)
    labels_alt: 비교 clustering labels (e.g., MedImp)
    index가 같고 순서가 맞는 1D array-like / pd.Series
    
    반환:
      - summary_df: 각 ref cluster별 best-matched alt cluster와 Jaccard
      - ctab: cross-tab
    """
    s_ref = pd.Series(labels_ref, name=ref_name).reset_index(drop=True)
    s_alt = pd.Series(labels_alt, name=alt_name).reset_index(drop=True)

    if len(s_ref) != len(s_alt):
        raise ValueError("labels_ref and labels_alt must have the same length.")

    # cross-tab
    ctab = pd.crosstab(s_ref, s_alt)

    rows = []
    for ref_cluster in ctab.index:
        # ref cluster와 가장 많이 겹치는 alt cluster 선택
        best_alt_cluster = ctab.loc[ref_cluster].idxmax()

        ref_idx = set(np.where(s_ref.values == ref_cluster)[0])
        alt_idx = set(np.where(s_alt.values == best_alt_cluster)[0])

        inter = len(ref_idx & alt_idx)
        union = len(ref_idx | alt_idx)
        jaccard = inter / union if union > 0 else np.nan

        rows.append({
            f'{ref_name}_cluster': ref_cluster,
            f'Best_{alt_name}_cluster': best_alt_cluster,
            'Intersection_n': inter,
            'Union_n': union,
            'Jaccard': jaccard,
            f'{ref_name}_cluster_size': len(ref_idx),
            f'{alt_name}_cluster_size': len(alt_idx)
        })

    summary_df = pd.DataFrame(rows).sort_values(by=f'{ref_name}_cluster').reset_index(drop=True)
    return summary_df, ctab

In [17]:
# Orig vs MedImp
jacc_orig_med, ctab_orig_med = clusterwise_jaccard(
    df['Orig_Cluster_Labels'],
    df['MedImp_Cluster_Labels'],
    ref_name='Orig',
    alt_name='MedImp'
)

print("[Cluster-wise Jaccard: Orig vs MedImp]")
jacc_orig_med

[Cluster-wise Jaccard: Orig vs MedImp]


,Orig_cluster,Best_MedImp_cluster,Intersection_n,Union_n,Jaccard,Orig_cluster_size,MedImp_cluster_size
0,1,2,230,413,0.556901,405,238
1,2,3,320,355,0.901408,322,353
2,3,4,157,368,0.426630,163,362
3,4,4,200,554,0.361011,392,362


In [18]:
# Orig vs NonImp
jacc_orig_non, ctab_orig_non = clusterwise_jaccard(
    df['Orig_Cluster_Labels'],
    df['NonImp_Cluster_Labels'],
    ref_name='Orig',
    alt_name='NonImp'
)

print("\n[Cluster-wise Jaccard: Orig vs NonImp]")
jacc_orig_non


[Cluster-wise Jaccard: Orig vs NonImp]


,Orig_cluster,Best_NonImp_cluster,Intersection_n,Union_n,Jaccard,Orig_cluster_size,NonImp_cluster_size
0,1,1,348,449,0.775056,405,392
1,2,2,312,340,0.917647,322,330
2,3,3,153,192,0.796875,163,182
3,4,4,321,449,0.714922,392,378


In [19]:
# Orig vs MICE-I10
jacc_orig_mice10, ctab_orig_mice10 = clusterwise_jaccard(
    df['Orig_Cluster_Labels'],
    df['MICEImp-I10_Cluster_Labels'],
    ref_name='Orig',
    alt_name='MICE-I10'
)
print("\n[Cluster-wise Jaccard: Orig vs MICE-I10]")
jacc_orig_mice10


[Cluster-wise Jaccard: Orig vs MICE-I10]


,Orig_cluster,Best_MICE-I10_cluster,Intersection_n,Union_n,Jaccard,Orig_cluster_size,MICE-I10_cluster_size
0,1,1,345,446,0.773543,405,386
1,2,2,317,334,0.949102,322,329
2,3,3,145,210,0.690476,163,192
3,4,4,307,460,0.667391,392,375


In [20]:
# Orig vs MICE-I20
jacc_orig_mice20, ctab_orig_mice20 = clusterwise_jaccard(
    df['Orig_Cluster_Labels'],
    df['MICEImp-I20_Cluster_Labels'],
    ref_name='Orig',
    alt_name='MICE-I20'
)
print("\n[Cluster-wise Jaccard: Orig vs MICE-I20]")
jacc_orig_mice20


[Cluster-wise Jaccard: Orig vs MICE-I20]


,Orig_cluster,Best_MICE-I20_cluster,Intersection_n,Union_n,Jaccard,Orig_cluster_size,MICE-I20_cluster_size
0,1,1,345,446,0.773543,405,386
1,2,2,317,334,0.949102,322,329
2,3,3,145,210,0.690476,163,192
3,4,4,307,460,0.667391,392,375


In [21]:
# Orig vs KNN-K5
jacc_orig_knn5, ctab_orig_knn5 = clusterwise_jaccard(
    df['Orig_Cluster_Labels'],
    df['KNNImp-K5_Cluster_Labels'],
    ref_name='Orig',
    alt_name='KNN-K5'
)
print("\n[Cluster-wise Jaccard: Orig vs KNN-K5]")
jacc_orig_knn5


[Cluster-wise Jaccard: Orig vs KNN-K5]


,Orig_cluster,Best_KNN-K5_cluster,Intersection_n,Union_n,Jaccard,Orig_cluster_size,KNN-K5_cluster_size
0,1,1,322,455,0.707692,405,372
1,2,2,310,337,0.919881,322,325
2,3,3,152,206,0.737864,163,195
3,4,4,295,487,0.605749,392,390


In [22]:
# Orig vs KNN-K10
jacc_orig_knn10, ctab_orig_knn10 = clusterwise_jaccard(
    df['Orig_Cluster_Labels'],
    df['KNNImp-K10_Cluster_Labels'],
    ref_name='Orig',
    alt_name='KNN-K10'
)
print("\n[Cluster-wise Jaccard: Orig vs KNN-K10]")
jacc_orig_knn10


[Cluster-wise Jaccard: Orig vs KNN-K10]


,Orig_cluster,Best_KNN-K10_cluster,Intersection_n,Union_n,Jaccard,Orig_cluster_size,KNN-K10_cluster_size
0,1,1,308,470,0.655319,405,373
1,2,2,314,355,0.884507,322,347
2,3,3,135,174,0.775862,163,146
3,4,4,299,509,0.587426,392,416


In [23]:
# Orig vs KNN-K10NW
jacc_orig_knn10nw, ctab_orig_knn10nw = clusterwise_jaccard(
    df['Orig_Cluster_Labels'],
    df['KNNImp-K10NW_Cluster_Labels'],
    ref_name='Orig',
    alt_name='KNN-K10NW'
)
print("\n[Cluster-wise Jaccard: Orig vs KNN-K10NW]")
jacc_orig_knn10nw


[Cluster-wise Jaccard: Orig vs KNN-K10NW]


,Orig_cluster,Best_KNN-K10NW_cluster,Intersection_n,Union_n,Jaccard,Orig_cluster_size,KNN-K10NW_cluster_size
0,1,1,383,564,0.679078,405,542
1,2,2,321,350,0.917143,322,349
2,3,3,136,174,0.781609,163,147
3,4,4,212,424,0.500000,392,244


In [24]:
# Orig vs KNN-K20
jacc_orig_knn20, ctab_orig_knn20 = clusterwise_jaccard(
    df['Orig_Cluster_Labels'],
    df['KNNImp-K20_Cluster_Labels'],
    ref_name='Orig',
    alt_name='KNN-K20'
)
print("\n[Cluster-wise Jaccard: Orig vs KNN-K20]")
jacc_orig_knn20


[Cluster-wise Jaccard: Orig vs KNN-K20]


,Orig_cluster,Best_KNN-K20_cluster,Intersection_n,Union_n,Jaccard,Orig_cluster_size,KNN-K20_cluster_size
0,1,1,370,543,0.681400,405,508
1,2,2,314,340,0.923529,322,332
2,3,3,154,207,0.743961,163,198
3,4,4,210,426,0.492958,392,244


In [9]:
## WCSS lists by Imps
#    k = 2 ~ 51 (length 50)
wcss_nonimp = [
    437965.4377627751, 143058.13699925502, 99533.15654320715, 86685.64445336035,
    80320.94780632836, 73562.97177608215, 68840.02873972207, 70554.29630861006,
    64302.38390335277, 61798.68863749522, 60141.263943107515, 59779.424872068106,
    58024.180136140036, 57996.45453524372, 56506.29838191042, 53687.55758537836,
    52972.61663221104, 52802.895513137315, 51571.490431514, 50842.0226507975,
    50114.71469375364, 50422.50386278708, 50467.25999356399, 49272.956282991196,
    48977.43024965094, 48408.391836027266, 48345.220099383965, 48201.80769191058,
    46771.17825364708, 47503.55752894752, 47057.16026362937, 47004.22172029998,
    46682.52537747448, 45896.055080242775, 45446.69882323513, 45023.29172965682,
    44953.864483474346, 44799.45604003551, 44769.000868653035, 44357.225779290144,
    44385.727232775665, 43984.23138918677, 43616.81142668638, 43121.64697503828,
    43746.316009264665, 43585.11037836961, 43395.58286060943, 43090.724626213705,
    42792.5724474682, 41614.37112031137
]

wcss_medimp = [
    128525.82084065153, 87359.87926777409, 75892.2478619446, 66059.52324776938,
    58487.480816893956, 53925.40108054531, 50489.55156630665, 49950.51209862824,
    47679.26854011864, 45193.102028320944, 42773.56081954185, 41159.589317792896,
    39872.88174166191, 39025.71132110192, 37742.72379787057, 37107.70774821809,
    36640.07702246372, 35538.665973163086, 35545.92779575567, 34531.182712305694,
    33817.386238792875, 33057.925859169576, 34697.78177158227, 32333.492787334057,
    31324.105255286853, 31675.1971664212, 30753.27732860685, 30462.599764632345,
    29718.885816747235, 29934.93532879435, 29619.75696581369, 29223.313827916187,
    28721.53335066373, 28485.387106704162, 28220.230690895103, 28028.05195890492,
    27711.103564783458, 27654.013170076756, 26984.544066917148, 27256.622639747962,
    27161.14350841713, 27065.95587080444, 26275.281718149756, 25992.72922281259,
    25990.674601790353, 25496.968399757297, 25368.908580130028, 25732.50957287384,
    25376.889900978033, 24747.182575960465
]

wcss_orig = [
    416936.51195827266, 127559.82816952295, 83622.69082728474, 71279.34163336708,
    64758.00809416406, 59111.88239223111, 53609.32960153667, 50202.994929858905,
    48527.73587803397, 45918.6534928139, 44184.565059059496, 42923.38420805446,
    41816.14252327288, 40203.09757402983, 40106.406433601216, 38743.08827337677,
    36912.963899322414, 36233.84460261688, 36250.76094386767, 34983.7559800555,
    34711.968484512996, 34235.89204322776, 33601.819783559804, 33309.32336152543,
    33085.2368026456, 32414.189342546502, 32138.591353015257, 32724.18493379876,
    30724.765713894652, 30482.081540763753, 30071.129502871376, 29748.978191965158,
    29514.921419261274, 29607.088340417646, 29359.011405589485, 29014.070244665538,
    30239.490508963776, 28283.86979553664, 28470.896031756893, 27716.129294486687,
    27982.828962729767, 27270.715501769326, 27050.872942524642, 26547.387634353618,
    26636.644105055093, 26729.376480018906, 26088.019573217443, 26253.452797227586,
    25932.181874806913, 25513.560383916276
]

k_values = list(range(2, 52))

wcss_df = pd.DataFrame({
    'k': k_values,
    'NonImp_WCSS': wcss_nonimp,
    'MedImp_WCSS': wcss_medimp,
    'Orig_WCSS': wcss_orig
})

print("\n[WCSS table]")
print(wcss_df)

# k=8 row
wcss_k8 = wcss_df.loc[wcss_df['k'] == 8].copy()
print("\n[WCSS at k=8]")
print(wcss_k8)


[WCSS table]
     k    NonImp_WCSS    MedImp_WCSS      Orig_WCSS
0    2  437965.437763  128525.820841  416936.511958
1    3  143058.136999   87359.879268  127559.828170
2    4   99533.156543   75892.247862   83622.690827
3    5   86685.644453   66059.523248   71279.341633
4    6   80320.947806   58487.480817   64758.008094
5    7   73562.971776   53925.401081   59111.882392
6    8   68840.028740   50489.551566   53609.329602
7    9   70554.296309   49950.512099   50202.994930
8   10   64302.383903   47679.268540   48527.735878
9   11   61798.688637   45193.102028   45918.653493
10  12   60141.263943   42773.560820   44184.565059
11  13   59779.424872   41159.589318   42923.384208
12  14   58024.180136   39872.881742   41816.142523
13  15   57996.454535   39025.711321   40203.097574
14  16   56506.298382   37742.723798   40106.406434
15  17   53687.557585   37107.707748   38743.088273
16  18   52972.616632   36640.077022   36912.963899
17  19   52802.895513   35538.665973   36233.84460

In [10]:
## Silhouette lists by Imps
silhouette_df = pd.DataFrame({
    'k': [2, 3, 4, 5, 6, 7, 8, 9],
    'NonImp_avg': [0.894, 0.230, 0.138, 0.141, 0.150, 0.164, 0.162, 0.146],
    'NonImp_std': [0.012, 0.334, 0.340, 0.350, 0.343, 0.316, 0.308, 0.300],
    'MedImp_avg': [0.229, 0.242, 0.211, 0.162, 0.142, 0.128, 0.128, 0.118],
    'MedImp_std': [0.095, 0.081, 0.081, 0.083, 0.092, 0.063, 0.068, 0.070],
    'Orig_avg': [0.893, 0.262, 0.228, 0.230, 0.165, 0.164, 0.146, 0.144],
    'Orig_std': [0.014, 0.317, 0.322, 0.289, 0.281, 0.273, 0.271, 0.255]
})

print("\n[Silhouette table]")
print(silhouette_df)

sil_k8 = silhouette_df.loc[silhouette_df['k'] == 8].copy()
print("\n[Silhouette @ k=8]")
print(sil_k8)


[Silhouette table]
   k  NonImp_avg  NonImp_std  MedImp_avg  MedImp_std  Orig_avg  Orig_std
0  2       0.894       0.012       0.229       0.095     0.893     0.014
1  3       0.230       0.334       0.242       0.081     0.262     0.317
2  4       0.138       0.340       0.211       0.081     0.228     0.322
3  5       0.141       0.350       0.162       0.083     0.230     0.289
4  6       0.150       0.343       0.142       0.092     0.165     0.281
5  7       0.164       0.316       0.128       0.063     0.164     0.273
6  8       0.162       0.308       0.128       0.068     0.146     0.271
7  9       0.146       0.300       0.118       0.070     0.144     0.255

[Silhouette @ k=8]
   k  NonImp_avg  NonImp_std  MedImp_avg  MedImp_std  Orig_avg  Orig_std
6  8       0.162       0.308       0.128       0.068     0.146     0.271


In [20]:
## Summary table for revision
summary_df = pd.DataFrame({
    'Model': ['Orig', 'NonImp', 'MedImp'],
    'PCs': [6, 6, 6],
    'Explained_variance_pct': [32.80, 32.91, 32.73],
    'WCSS_k8': [
        float(wcss_df.loc[wcss_df['k'] == 8, 'Orig_WCSS'].iloc[0]),
        float(wcss_df.loc[wcss_df['k'] == 8, 'NonImp_WCSS'].iloc[0]),
        float(wcss_df.loc[wcss_df['k'] == 8, 'MedImp_WCSS'].iloc[0]),
    ],
    'Silhouette_avg_k8': [
        float(silhouette_df.loc[silhouette_df['k'] == 8, 'Orig_avg'].iloc[0]),
        float(silhouette_df.loc[silhouette_df['k'] == 8, 'NonImp_avg'].iloc[0]),
        float(silhouette_df.loc[silhouette_df['k'] == 8, 'MedImp_avg'].iloc[0]),
    ],
    'Silhouette_std_k8': [
        float(silhouette_df.loc[silhouette_df['k'] == 8, 'Orig_std'].iloc[0]),
        float(silhouette_df.loc[silhouette_df['k'] == 8, 'NonImp_std'].iloc[0]),
        float(silhouette_df.loc[silhouette_df['k'] == 8, 'MedImp_std'].iloc[0]),
    ],
    'ARI_vs_Orig': [
        1.0,
        ari_orig_non,
        ari_orig_med
    ]
})

print("[Summary table]")
summary_df


[Summary table]


,Model,PCs,Explained_variance_pct,WCSS_k8,Silhouette_avg_k8,Silhouette_std_k8,ARI_vs_Orig
0,Orig,6,32.80,53609.329602,0.146,0.271,1.000000
1,NonImp,6,32.91,68840.028740,0.162,0.308,0.397395
2,MedImp,6,32.73,50489.551566,0.128,0.068,0.397011


In [ ]:
# Save all outputs
ari_df.to_csv(os.path.join(base_dir, 'ARI_by_Imps.csv'), index=False, encoding='utf-8-sig')
cluster_sizes.to_csv(os.path.join(base_dir, 'ClusterSizes_by_Imps.csv'), encoding='utf-8-sig')
ct_orig_med.to_csv(os.path.join(base_dir, 'Crosstab_Orig_vs_MedImp.csv'), encoding='utf-8-sig')
ct_orig_non.to_csv(os.path.join(base_dir, 'Crosstab_Orig_vs_NonImp.csv'), encoding='utf-8-sig')
ct_med_non.to_csv(os.path.join(base_dir, 'Crosstab_MedImp_vs_NonImp.csv'), encoding='utf-8-sig')
wcss_df.to_csv(os.path.join(base_dir, 'WCSS_by_Imps.csv'), index=False, encoding='utf-8-sig')
silhouette_df.to_csv(os.path.join(base_dir, 'Silhouette_by_Imps.csv'), index=False, encoding='utf-8-sig')
summary_df.to_csv(os.path.join(base_dir, 'Summary_Imputation_Comparison.csv'), index=False, encoding='utf-8-sig')

print("\nSaved all comparison files to:")
print(base_dir)